# v7 — XGBoost + LightGBM blend

**What v7 adds.** A LightGBM model per fold next to the XGBoost one; the two probability vectors are averaged 50/50. Features are v6's (`geo_cluster`, cleaned text, interactions). LightGBM reuses XGBoost's searched hyperparameters through a name mapping (no search of its own). v7 still trains with **balanced weights**.

**Hypotheses.**
- **H8** — a blend beats either model alone, because different model families make different errors. The v7 doc says explicitly to check *`blend > max(xgb, lgbm)`*, not only `blend > v6`.
- **H5⁵** — weights cost accuracy (confirmed at v2–v6). Main protocol is **no weights**.
- **H9 (new)** — the "same" hyperparameters do not describe the same model in LightGBM. XGBoost grows depth-limited trees (`max_depth=12`), while LightGBM's `num_leaves` defaults to 31, which caps every tree far below what depth 12 allows. If so, v7's LightGBM is a *much shallower* model than its XGBoost partner, and a fairer partner is worth testing.

**Protocol.** One pass per configuration. Inside each fold, features are fit on the training rows, and **both** models early-stop honestly on a 10% holdout carved from the training rows (v7 stops on the scored fold; we run that too, as its own row). OOF probabilities are saved, so single-model and blend accuracy come from the **same** fits.

| row | what |
|---|---|
| ref | v6 row 6a: XGBoost alone, 300 trees, no weights (0.8108) |
| 7a | **no weights, honest ES**: XGBoost / LightGBM (v7's mapping) / 50-50 blend |
| 7b | as 7a, but LightGBM with `num_leaves=255` (H9) |
| 7c | **v7 as shipped**: ES on the scored fold + **weights**, v7's LightGBM mapping |

In [1]:
import sys, warnings, json
sys.path.insert(0, '.')
warnings.simplefilter('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import *
pd.set_option('display.width', 140)
import lightgbm as lgb, xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight

df, y, le = get_train('v7')
Flex = make_flex('v7')
BEST = json.loads(open('cache/v3_search.json').read())['best_params']
ref = next(r for r in map(json.loads, RESULTS.read_text().splitlines()) if r['label'].startswith('6a '))
print(f"reference 6a (v6 features, XGB alone, no weights): {ref['mean']:.4f} ± {ref['std']:.4f}")
KW = dict(k=30, text=dict(cols=['funder', 'installer'], min=20), inter=('qw', 'pm', 'pop'))
LP = dict(max_depth=BEST['max_depth'], learning_rate=BEST['learning_rate'], feature_fraction=BEST['colsample_bytree'],
          bagging_fraction=BEST['subsample'], bagging_freq=1, min_child_weight=BEST['min_child_weight'])   # = v7's build_lgbm_params
print('LightGBM params (v7 mapping):', LP, '| num_leaves default = 31')

def cv_blend(balanced=False, es='inner', lgbm_extra=None, rounds=30, max_n=2000):
    """One CV pass: per fold fit XGBoost and LightGBM on the same features, early-stop both, keep OOF probabilities."""
    X_all = df.drop(columns=[TARGET])
    px, pl = np.zeros((len(y), 3)), np.zeros((len(y), 3)); it = {'xgb': [], 'lgbm': []}
    for tr, va in folds(y):
        fp = Flex(**KW); Xtr, Xva = fp.fit_transform(X_all.iloc[tr]), fp.transform(X_all.iloc[va]); ytr = y[tr]
        if es == 'val':
            fx, fy, ex, ey = Xtr, ytr, Xva, y[va]
        else:
            a, b = train_test_split(np.arange(len(ytr)), test_size=0.1, stratify=ytr, random_state=SEED)
            fx, fy, ex, ey = Xtr.iloc[a], ytr[a], Xtr.iloc[b], ytr[b]
        w = compute_sample_weight('balanced', fy) if balanced else None
        mx = xgb.XGBClassifier(**BEST, n_estimators=max_n, early_stopping_rounds=rounds, eval_metric='mlogloss',
                               tree_method='hist', enable_categorical=True, random_state=SEED, n_jobs=-1)
        mx.fit(fx, fy, sample_weight=w, eval_set=[(ex, ey)], verbose=False)
        ml = lgb.LGBMClassifier(**LP, **(lgbm_extra or {}), n_estimators=max_n, random_state=SEED, n_jobs=-1, verbosity=-1)
        ml.fit(fx, fy, sample_weight=w, eval_set=[(ex, ey)], callbacks=[lgb.early_stopping(rounds, verbose=False)])
        px[va], pl[va] = mx.predict_proba(Xva), ml.predict_proba(Xva)
        it['xgb'].append(mx.best_iteration + 1); it['lgbm'].append(ml.best_iteration_)
    return px, pl, it

def score(tag, px, pl, it, note=''):
    out = {}
    for name, p in [('xgb', px), ('lgbm', pl), ('blend', (px + pl) / 2)]:
        sc = [(p[va].argmax(1) == y[va]).mean() for _, va in folds(y)]
        r = log_result('v7', f'{tag} {name}', sc, note=f'trees/fold {it.get(name, "")}' if name != 'blend' else note)
        d = np.array(sc) - np.array(ref['folds']); out[name] = sc
        print(f"{tag} {name:<6} {r['mean']:.4f} ± {r['std']:.4f}   Δ vs ref {d.mean():+.4f}")
    b, best = np.array(out['blend']), np.maximum(out['xgb'], out['lgbm'])
    print(f"   blend - max(xgb, lgbm), per fold: {np.round(b - best, 4).tolist()}  (mean {np.mean(b - best):+.4f})")
    return out

reference 6a (v6 features, XGB alone, no weights): 0.8108 ± 0.0039
LightGBM params (v7 mapping): {'max_depth': 12, 'learning_rate': 0.03, 'feature_fraction': 0.7, 'bagging_fraction': 0.6, 'bagging_freq': 1, 'min_child_weight': 5} | num_leaves default = 31


## 7a — no weights, honest early stopping, v7's LightGBM mapping

In [2]:
px_a, pl_a, it_a = cv_blend()
s_a = score('7a', px_a, pl_a, it_a)

7a xgb    0.8098 ± 0.0048   Δ vs ref -0.0011
7a lgbm   0.7993 ± 0.0043   Δ vs ref -0.0116
7a blend  0.8081 ± 0.0046   Δ vs ref -0.0027
   blend - max(xgb, lgbm), per fold: [-0.0013, -0.0027, -0.0032, 0.0008, -0.0017]  (mean -0.0016)


## Why blending can or can't help: how different are the two models?

In [3]:
pa, la = px_a.argmax(1), pl_a.argmax(1)
bl = ((px_a + pl_a) / 2).argmax(1)
print('rows where xgb and lgbm disagree      :', f'{(pa != la).mean():.1%}')
print('accuracy when they agree              :', f'{(y[pa == la] == pa[pa == la]).mean():.4f}')
print('oracle (right if EITHER is right)     :', f'{((pa == y) | (la == y)).mean():.4f}   <- ceiling for any blend of the two')
print('xgb / lgbm / blend OOF accuracy       :', f'{(pa == y).mean():.4f} / {(la == y).mean():.4f} / {(bl == y).mean():.4f}')
print('trees chosen: xgb', it_a['xgb'], '| lgbm', it_a['lgbm'])

rows where xgb and lgbm disagree      : 7.1%
accuracy when they agree              : 0.8314
oracle (right if EITHER is right)     : 0.8367   <- ceiling for any blend of the two
xgb / lgbm / blend OOF accuracy       : 0.8098 / 0.7993 / 0.8081
trees chosen: xgb [401, 297, 324, 377, 306] | lgbm [685, 480, 420, 413, 370]


## 7b — H9: give LightGBM trees as complex as XGBoost's (`num_leaves=255`)

In [4]:
px_b, pl_b, it_b = cv_blend(lgbm_extra=dict(num_leaves=255))
s_b = score('7b', px_b, pl_b, it_b, note='lgbm num_leaves=255')

7b xgb    0.8098 ± 0.0048   Δ vs ref -0.0011
7b lgbm   0.8049 ± 0.0041   Δ vs ref -0.0060
7b blend  0.8105 ± 0.0050   Δ vs ref -0.0004
   blend - max(xgb, lgbm), per fold: [0.0013, 0.0007, -0.0008, 0.0012, 0.0011]  (mean +0.0007)


## 7c — v7 as shipped: early stopping on the scored fold, balanced weights, v7's LightGBM mapping

In [5]:
px_c, pl_c, it_c = cv_blend(balanced=True, es='val')
s_c = score('7c', px_c, pl_c, it_c, note='as shipped')

7c xgb    0.7817 ± 0.0056   Δ vs ref -0.0292
7c lgbm   0.7640 ± 0.0063   Δ vs ref -0.0469
7c blend  0.7800 ± 0.0066   Δ vs ref -0.0308
   blend - max(xgb, lgbm), per fold: [0.0003, -0.0019, -0.0018, -0.0026, -0.0025]  (mean -0.0017)


## Results

In [6]:
res = pd.DataFrame([r for r in map(json.loads, RESULTS.read_text().splitlines()) if r['version'] == 'v7'])
res['acc'] = res['mean'].round(4); res['fold std'] = res['std'].round(4)
res['Δ vs ref'] = (res['mean'] - ref['mean']).round(4)
res[['label', 'acc', 'fold std', 'Δ vs ref']].set_index('label')

,acc,fold std,Δ vs ref
label,,,
7a xgb,0.8098,0.0048,-0.0011
7a lgbm,0.7993,0.0043,-0.0116
7a blend,0.8081,0.0046,-0.0027
7b xgb,0.8098,0.0048,-0.0011
7b lgbm,0.8049,0.0041,-0.0060
7b blend,0.8105,0.0050,-0.0004
7c xgb,0.7817,0.0056,-0.0292
7c lgbm,0.7640,0.0063,-0.0469
7c blend,0.7800,0.0066,-0.0308


## Conclusion

Reference = v6 row 6a (XGBoost alone, fixed 300 trees, no weights) **0.8108 ± 0.0036**. All v7 rows use honest early stopping unless marked.

| row | XGBoost | LightGBM | 50/50 blend | blend − best single |
|---|---|---|---|---|
| 7a no weights, v7's LightGBM mapping | 0.8098 | 0.7993 | 0.8081 | **−0.0016** (4 of 5 folds negative) |
| 7b no weights, LightGBM `num_leaves=255` | 0.8098 | 0.8049 | **0.8105** | +0.0007 (4 of 5 folds positive) |
| **7c v7 as shipped** (val-fold ES + weights) | 0.7817 | 0.7640 | **0.7800** | **−0.0017** |

- **H8 (blend beats either model alone) is not supported.** v7's own doc says to check `blend > max(xgb, lgbm)`. As shipped, that check **fails**: the blend (0.7800) is *below* XGBoost alone (0.7817). Without weights it also fails at v7's mapping (0.8081 vs 0.8098). Only with a stronger LightGBM does the blend edge ahead (+0.0007), about a sixth of the fold std, so not distinguishable from noise.
- **H9 (the "same" hyperparameters are not the same model) is supported.** Under v7's mapping LightGBM scores **0.7993, about 0.0105 below XGBoost**, because LightGBM's default `num_leaves=31` caps every tree far below what `max_depth=12` allows in XGBoost. Raising it to 255 lifts LightGBM by +0.0056 (0.8049) and turns the blend from a loss into a tie. The v7 doc calls the mapping a "proxy"; it is in fact a much weaker model than its partner.
- **Why there is so little to gain.** The two models disagree on only **7.1%** of rows, and when they agree they are right 83.1% of the time. Even an oracle that always picks the correct model would reach just **0.8367**, which is +0.027 over XGBoost, and a 50/50 average cannot get near that.
- **Weights, a sixth time.** v7 as shipped scores about 0.78 on every model, 0.03 below the same code without weights. LightGBM is hit hardest (0.7640 vs 0.7993).
- **Nothing in v7 beats the earlier honest best.** The best v7 row (7b blend, 0.8105) is below v5's 0.8115 and v3's 0.8110, all within noise.

**Caveats.** One seed, v3's params reused for both models, and LightGBM was never searched (a proper search might help it, which is the doc's own caveat). 7b's `num_leaves=255` was a single guess, not tuned. CV only, no leaderboard.

**Overall:** v7 is the first version where the result is reportable as *negative at both the shipped and the honest setting*: the blend does not beat XGBoost alone. See `98_features` and `99_comparison` for the whole v1–v7 picture.